# Выполнение ЛР №5: Классификация и регрессия

## Подключение библиотек

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Импорт модулей sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Дополнительные импорты для обработки данных
import warnings
warnings.filterwarnings('ignore')

## Настройка библиотек

In [ ]:
# Настройка стилей визуализации
plt.style.use('default')
sns.set_palette("husl")

# Настройка параметров отображения
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Настройка pandas для отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Настройка numpy для воспроизводимости результатов
np.random.seed(42)

print("Библиотеки успешно импортированы и настроены!")
print(f"Версия pandas: {pd.__version__}")
print(f"Версия numpy: {np.__version__}")
print(f"Версия matplotlib: {plt.matplotlib.__version__}")
print(f"Версия seaborn: {sns.__version__}")

## Задание 1: Классификация kNN на датасете flame

### Формулировка

Выполнить классификацию методом k ближайших соседей на датасете flame:
1. Загрузить и разобрать данные из файла flame.txt
2. Оценить точность для разных значений k от 2 до 20 с использованием кросс-валидации
3. Построить график зависимости точности от k

### Решение

#### 1.1 Загрузка и парсинг данных flame

In [ ]:
# Загрузка данных из файла flame.txt
flame_data_path = '../ЛР (4)/Вариант 4/flame.txt'

# Чтение данных с разделителем табуляция
flame_data = pd.read_csv(flame_data_path, sep='\t', header=None, names=['x1', 'x2', 'class'])

print("Данные flame успешно загружены!")
print(f"Размер датасета: {flame_data.shape}")
print("\nПервые 10 строк:")
print(flame_data.head(10))

print("\nИнформация о данных:")
print(flame_data.info())

print("\nРаспределение классов:")
print(flame_data['class'].value_counts().sort_index())

# Разделение данных на признаки (X) и метки классов (y)
X_flame = flame_data[['x1', 'x2']].values
y_flame = flame_data['class'].values

print(f"\nРазмер матрицы признаков X: {X_flame.shape}")
print(f"Размер вектора меток y: {y_flame.shape}")
print(f"Уникальные классы: {np.unique(y_flame)}")

#### 1.2 Оценка точности kNN для разных значений k

In [ ]:
# Оценка точности kNN для разных значений k от 2 до 20
k_range = range(2, 21)  # k от 2 до 20
k_scores = []

print("Оценка точности kNN для разных значений k:")
print("k\tТочность (среднее)\tСтандартное отклонение")
print("-" * 50)

# Цикл оценки для каждого k
for k in k_range:
    # Создание модели kNN
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # Кросс-валидация с 5 фолдами
    cv_scores = cross_val_score(knn, X_flame, y_flame, cv=5, scoring='accuracy')
    
    # Сохранение среднего значения точности
    mean_accuracy = cv_scores.mean()
    std_accuracy = cv_scores.std()
    k_scores.append(mean_accuracy)
    
    print(f"{k}\t{mean_accuracy:.4f}\t\t{std_accuracy:.4f}")

print(f"\nВсего оценено k значений: {len(k_scores)}")
print(f"Лучшая точность: {max(k_scores):.4f} при k = {k_range[k_scores.index(max(k_scores))]}")
print(f"Худшая точность: {min(k_scores):.4f} при k = {k_range[k_scores.index(min(k_scores))]}")

#### 1.3 Построение графика точности vs k

In [ ]:
# Построение графика зависимости точности от k
plt.figure(figsize=(12, 8))

# Основной график
plt.plot(k_range, k_scores, 'bo-', linewidth=2, markersize=8, label='Точность кросс-валидации')

# Выделение максимального значения
best_k = k_range[k_scores.index(max(k_scores))]
best_score = max(k_scores)
plt.plot(best_k, best_score, 'ro', markersize=12, label=f'Лучший результат (k={best_k})')

# Настройка графика
plt.xlabel('Количество соседей (k)', fontsize=14)
plt.ylabel('Точность классификации', fontsize=14)
plt.title('Зависимость точности kNN от количества соседей k\n(датасет flame)', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Установка диапазона осей
plt.xlim(1.5, 20.5)
plt.ylim(min(k_scores) - 0.02, max(k_scores) + 0.02)

# Добавление аннотации для лучшего результата
plt.annotate(f'Максимум: {best_score:.4f}', 
             xy=(best_k, best_score), 
             xytext=(best_k + 2, best_score + 0.01),
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
             fontsize=12, color='red')

# Настройка тиков на оси x
plt.xticks(range(2, 21, 2))

plt.tight_layout()
plt.show()

# Вывод статистики
print(f"\nСтатистика по результатам:")
print(f"Оптимальное значение k: {best_k}")
print(f"Максимальная точность: {best_score:.4f}")
print(f"Средняя точность по всем k: {np.mean(k_scores):.4f}")
print(f"Стандартное отклонение: {np.std(k_scores):.4f}")

#### Выводы по заданию 1

1. **Загрузка данных**: Успешно загружен датасет flame с двумя признаками (x1, x2) и двумя классами (1, 2)
2. **Оценка kNN**: Проведена оценка точности для k от 2 до 20 с использованием 5-фолдовой кросс-валидации
3. **Визуализация**: Построен график зависимости точности от k, который показывает оптимальное значение k
4. **Результат**: Определено оптимальное значение k для данного датасета

## Задание 2: Визуализация данных с разделением на классы

### Формулировка

Визуализировать обучающие и тестовые данные с метками классов:
1. Разделить данные flame на обучающий и тестовый наборы
2. Создать диаграмму рассеяния с цветовым кодированием по классам
3. Различить обучающие и тестовые данные визуально
4. Добавить легенду с метками классов

### Решение

#### 2.1 Разделение данных на обучающий и тестовый наборы

In [ ]:
# Разделение данных flame на обучающий и тестовый наборы
# Используем фиксированный random_state для воспроизводимости результатов
X_train, X_test, y_train, y_test = train_test_split(
    X_flame, y_flame, 
    test_size=0.3,  # 30% данных для тестирования
    random_state=42,  # Фиксированное значение для воспроизводимости
    stratify=y_flame  # Сохранение пропорций классов
)

print("Данные успешно разделены на обучающий и тестовый наборы!")
print(f"Размер исходного датасета: {X_flame.shape[0]} образцов")
print(f"Размер обучающего набора: {X_train.shape[0]} образцов ({X_train.shape[0]/X_flame.shape[0]*100:.1f}%)")
print(f"Размер тестового набора: {X_test.shape[0]} образцов ({X_test.shape[0]/X_flame.shape[0]*100:.1f}%)")

print("\nРаспределение классов в обучающем наборе:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for class_label, count in zip(unique_train, counts_train):
    print(f"  Класс {class_label}: {count} образцов ({count/len(y_train)*100:.1f}%)")

print("\nРаспределение классов в тестовом наборе:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for class_label, count in zip(unique_test, counts_test):
    print(f"  Класс {class_label}: {count} образцов ({count/len(y_test)*100:.1f}%)")

# Проверка корректности разделения
total_samples = X_train.shape[0] + X_test.shape[0]
print(f"\nПроверка: {X_train.shape[0]} + {X_test.shape[0]} = {total_samples} (исходно: {X_flame.shape[0]})")
assert total_samples == X_flame.shape[0], "Ошибка: потеря данных при разделении!"

#### 2.2 Создание диаграммы рассеяния с классами

In [ ]:
# Создание диаграммы рассеяния с цветовым кодированием по классам
plt.figure(figsize=(14, 10))

# Определение цветов для классов
colors = {1: 'red', 2: 'blue'}
class_names = {1: 'Класс 1', 2: 'Класс 2'}

# Создание subplot для лучшего размещения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# График 1: Обучающие и тестовые данные отдельно
# Обучающие данные
for class_label in np.unique(y_train):
    mask_train = y_train == class_label
    ax1.scatter(X_train[mask_train, 0], X_train[mask_train, 1], 
               c=colors[class_label], alpha=0.7, s=60, 
               label=f'{class_names[class_label]} (обучение)', 
               marker='o', edgecolors='black', linewidth=0.5)

# Тестовые данные
for class_label in np.unique(y_test):
    mask_test = y_test == class_label
    ax1.scatter(X_test[mask_test, 0], X_test[mask_test, 1], 
               c=colors[class_label], alpha=0.9, s=80, 
               label=f'{class_names[class_label]} (тест)', 
               marker='^', edgecolors='black', linewidth=1)

ax1.set_xlabel('Признак x1', fontsize=12)
ax1.set_ylabel('Признак x2', fontsize=12)
ax1.set_title('Разделение данных flame на обучающий и тестовый наборы', fontsize=14)
ax1.legend(fontsize=10, loc='upper right')
ax1.grid(True, alpha=0.3)

# График 2: Все данные с выделением типа набора
# Обучающие данные
ax2.scatter(X_train[:, 0], X_train[:, 1], 
           c=[colors[label] for label in y_train], 
           alpha=0.6, s=50, label='Обучающие данные', 
           marker='o', edgecolors='gray', linewidth=0.3)

# Тестовые данные
ax2.scatter(X_test[:, 0], X_test[:, 1], 
           c=[colors[label] for label in y_test], 
           alpha=1.0, s=80, label='Тестовые данные', 
           marker='^', edgecolors='black', linewidth=1)

ax2.set_xlabel('Признак x1', fontsize=12)
ax2.set_ylabel('Признак x2', fontsize=12)
ax2.set_title('Визуализация всех данных flame по типу набора', fontsize=14)
ax2.legend(fontsize=10, loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Дополнительная детальная визуализация
plt.figure(figsize=(12, 8))

# Создание более детального графика с четким разделением
for class_label in np.unique(y_flame):
    # Обучающие данные для каждого класса
    mask_train = y_train == class_label
    plt.scatter(X_train[mask_train, 0], X_train[mask_train, 1], 
               c=colors[class_label], alpha=0.7, s=70, 
               label=f'{class_names[class_label]} - Обучение ({np.sum(mask_train)} точек)', 
               marker='o', edgecolors='darkgray', linewidth=0.8)
    
    # Тестовые данные для каждого класса
    mask_test = y_test == class_label
    plt.scatter(X_test[mask_test, 0], X_test[mask_test, 1], 
               c=colors[class_label], alpha=1.0, s=100, 
               label=f'{class_names[class_label]} - Тест ({np.sum(mask_test)} точек)', 
               marker='^', edgecolors='black', linewidth=1.2)

plt.xlabel('Признак x1', fontsize=14)
plt.ylabel('Признак x2', fontsize=14)
plt.title('Детальная визуализация данных flame с разделением на классы и наборы', fontsize=16)
plt.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.grid(True, alpha=0.3)

# Добавление статистики на график
stats_text = f"""Статистика разделения:
Всего образцов: {len(X_flame)}
Обучающий набор: {len(X_train)} ({len(X_train)/len(X_flame)*100:.1f}%)
Тестовый набор: {len(X_test)} ({len(X_test)/len(X_flame)*100:.1f}%)
Классы: {len(np.unique(y_flame))}"""

plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes, 
         fontsize=10, verticalalignment='top', 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

# Вывод дополнительной информации
print("\nВизуализация данных завершена!")
print("\nОписание графиков:")
print("1. Первый график: Показывает разделение данных по классам и типу набора")
print("2. Второй график: Показывает все данные с выделением обучающих и тестовых")
print("3. Третий график: Детальная визуализация с полной статистикой")
print("\nОбозначения:")
print("- Круги (o): Обучающие данные")
print("- Треугольники (^): Тестовые данные")
print("- Красный цвет: Класс 1")
print("- Синий цвет: Класс 2")

#### Выводы по заданию 2

1. **Разделение данных**: Данные flame успешно разделены на обучающий (70%) и тестовый (30%) наборы с сохранением пропорций классов
2. **Визуализация классов**: Создана диаграмма рассеяния с цветовым кодированием по классам (красный - класс 1, синий - класс 2)
3. **Различение наборов**: Обучающие данные показаны кругами, тестовые - треугольниками для четкого визуального различения
4. **Легенда и статистика**: Добавлены подробные легенды и статистическая информация о распределении данных
5. **Воспроизводимость**: Использован фиксированный random_state=42 для обеспечения воспроизводимости результатов

## Задание 3

### Формулировка

### Решение

## Задание 4

### Формулировка

### Решение

## Задание 5

### Формулировка

### Решение

## Задание 6

### Формулировка

### Решение

## Задание 7

### Формулировка

### Решение

## Задание 8

### Формулировка

### Решение